# Final LOSO Evaluation: Results Overview

Loads the already-computed leave-one-slide-out (LOSO) result CSVs under
`data/combined/03_integration/autoencoders/`, produced by the scripts in `scripts/`, and shows them as
summary tables. 

Run each script below first if its CSV is missing (see `scripts/README` usage docstrings):
`evaluate_clustering_quality.py`, `evaluate_k_matched_ari.py --smooth-hops 1 2 3`,
`evaluate_disease_signal_ground_truth.py`, `evaluate_k_matched_ari.py --baselines`.

Sections: retrieval, clustering quality before/after cluster-label smoothing, clustering quality before/after
embedding smoothing, disease prediction (per-unit, against the real ground truth), baseline comparison,
embedding agreement.

## Setup

In [ ]:
import sys
from pathlib import Path
import glob

sys.path.insert(0, "scripts")
import pandas as pd

import evaluate_clustering_quality as ecq

BASE = ecq.BASEDIR / "combined/03_integration/autoencoders"
UNIT_LABEL = {"nuclei": "Nuclei", "bins": "Bins"}
VARIANT_LABEL = {"svg": "SVG", "colon_xenium_panel": "Xenium panel", "svg_semantic_excl_raw_0.2": "SVG + semantic excl."}

pd.options.display.float_format = "{:.3f}".format
pd.options.display.width = 160


def load_csvs(pattern, dedupe_cols=None):
    """Reads every CSV matching `pattern` under BASE (recursively) and concatenates them. Several of the
    pipeline's CSVs exist both as a merged file and as its own now-redundant shards on disk at the same time,
    so this always de-duplicates on `dedupe_cols` (keep="last") rather than assume only one or the other is
    present."""
    paths = sorted(BASE.glob(pattern)) + sorted(BASE.glob(f"*/{pattern}"))
    if not paths:
        print(f"no files matched {pattern!r} under {BASE}")
        return pd.DataFrame()
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True, sort=False)
    if dedupe_cols:
        df = df.drop_duplicates(subset=dedupe_cols, keep="last")
    return df

## Retrieval (cell &harr; token, train vs. held-out)

One row per finished LOSO fold. `cfg__*` columns carry that fold's exact settings, which is how the unit and
gene-set variant are recovered here instead of parsing filenames.

In [ ]:
retrieval = load_csvs("loso_token_log_*.csv")
retrieval["unit"] = retrieval["cfg__bin_size"].map({"002um": "nuclei", "008um": "bins"})
retrieval["variant"] = retrieval["cfg__local_anchor_sim_threshold"].notna().map({True: "svg_semantic_excl_raw_0.2"}).fillna(
    retrieval["cfg__gene_panel_path"].fillna("").str.contains("colon_xenium").map({True: "colon_xenium_panel"}).fillna("svg")
)
retrieval = retrieval.drop_duplicates(subset=["unit", "variant", "held_out_slide"], keep="last")

cols = ["held_out_cell_top1_retrieval", "held_out_cell_cluster_top1_retrieval",
        "train_cell_top1_retrieval", "train_cell_cluster_top1_retrieval", "combined_token_leakage_mean"]

print(f"{len(retrieval)} finished folds\n")
print("Mean over held-out slides, per unit / variant:")
retrieval.groupby(["unit", "variant"])[cols].mean().round(3)

In [ ]:
# per-slide detail
retrieval.set_index(["unit", "variant", "held_out_slide"])[cols].sort_index().round(3)

## Clustering quality: ARI / NMI, before vs. after cluster-label smoothing

`refine_domains` reassigns each unit's Leiden label by a majority vote of its spatial neighbours, after
clustering. `label_kind`: `raw` = unsmoothed, `hard` = unconditional majority vote, `spagcn` = SpaGCN's
conditional rule. `reference_kind`: `raw`/`smoothed` = label-transfer (unsmoothed / spatially smoothed
reference), `gt` = the pathologist annotation (`CRC_P2` only).

In [ ]:
# "clustering_quality.csv" is what evaluate_clustering_quality.py writes directly (current);
# "final_loso_clustering_quality*.csv" is the old merged-shards name from the now-removed
# run_final_evaluation.py/merge_clustering_quality.py pipeline -- matched too, so this keeps
# working against older result directories without silently going stale on a fresh run.
quality = pd.concat([
    load_csvs("clustering_quality.csv"),
    load_csvs("final_loso_clustering_quality_smoothed.csv"),
], ignore_index=True, sort=False).drop_duplicates(
    subset=["unit", "variant", "held_out_slide", "embedding", "label_kind", "reference_kind"], keep="last")

summary = (quality.groupby(["unit", "variant", "embedding", "label_kind", "reference_kind"])
           [["ari_vs_label_transfer", "nmi_vs_label_transfer"]].mean())
summary.unstack("label_kind").round(3)

## Clustering quality: ARI / NMI, before vs. after embedding smoothing

A different kind of smoothing: each unit's *embedding* (not its label) is averaged with its spatial neighbours
for `h = 1, 2, 3` rounds, then re-clustered (`evaluate_k_matched_ari.py --smooth-hops`). Filtered to the raw
partition (`partition == "raw"`, no cluster-label smoothing on top) and the cluster-count-matched window
(7-10 clusters) used throughout, so this isolates the embedding-smoothing effect on its own.

In [ ]:
kmatched = load_csvs("final_loso_kmatched_ari*.csv",
                      dedupe_cols=["unit", "variant", "held_out_slide", "embedding", "resolution", "partition"])
matched = kmatched[(kmatched["partition"] == "raw") & kmatched["k"].between(7, 10)].copy()

hop = matched["embedding"].str.extract(r"_s(\d)$")[0]
matched["hops"] = hop.fillna(0).astype(int)
matched["embedding_base"] = matched["embedding"].str.replace(r"_s\d$", "", regex=True)

cols = ["ari_lt", "ari_sr", "ari_gt", "nmi_lt", "nmi_sr", "nmi_gt"]
summary = matched.groupby(["unit", "variant", "embedding_base", "hops"])[cols].mean()
summary.round(3)

## Disease prediction (per-unit, against the real ground truth)

Per-unit binary classification on `CRC_P2` (Neoplasm = diseased, everything else = normal),
via `evaluate_disease_signal_ground_truth.py`. Not the slide-level probe, which labelled every unit of a tumour
slide "diseased" even though only the annotated Neoplasm units actually are.

In [ ]:
disease = load_csvs("final_loso_disease_signal_ground_truth.csv",
                     dedupe_cols=["unit", "variant", "embedding", "class_weight"])
cols = ["accuracy", "balanced_accuracy", "recall_diseased", "specificity_normal", "auc"]
disease.set_index(["unit", "variant", "embedding", "class_weight"])[cols].sort_index().round(3)

## Baseline comparison (SpaGCN / STAGATE / SpatialFusion)

Each baseline's own mean ARI/NMI (`k = 7..10`, its own available slides) next to Genomosaic's, at the same
matched cluster-count window. Slide sets differ slightly between methods (not every baseline has output for
every slide), so this is a quick side-by-side, not the slide-matched comparison the thesis tables build.

In [ ]:
baselines = load_csvs("final_loso_baselines_ours_units*.csv",
                       dedupe_cols=["method", "held_out_slide", "k", "label_kind"])
cols = ["ari_lt", "ari_sr", "ari_gt", "nmi_lt", "nmi_sr", "nmi_gt"]

base_summary = baselines[baselines["k"].between(7, 10)].groupby("method")[cols].mean()
ours_summary = matched.groupby(["unit", "variant", "embedding_base"])[cols].mean().loc[
    pd.IndexSlice[:, :, "z_trans"]
].reset_index().assign(method=lambda d: "Genomosaic (" + d["unit"] + ", " + d["variant"] + ")").set_index("method")[cols]

pd.concat([base_summary, ours_summary]).round(3)

## Embedding agreement (z_trans vs. z_token vs. z_joint)

How much the transcriptomic, histology and joint-embedding partitions agree with each other (same slide, same
fold): does the joint embedding actually combine information from both, or mostly track one of them?

In [ ]:
agreement = load_csvs("final_loso_partition_agreement*.csv",
                       dedupe_cols=["unit", "variant", "held_out_slide", "emb_a", "emb_b"])
cols = ["ari", "nmi", "homogeneity_of_b_given_a", "completeness_of_b_given_a"]
agreement.groupby(["unit", "variant", "emb_a", "emb_b"])[cols].mean().round(3)